# Carte de risque — choix de la date
Lecture directe depuis DuckDB + parquets (pas d'API).

In [ ]:
# ── Paramètre à modifier ─────────────────────────────────────
DATE = "2025-12-15"   # format YYYY-MM-DD
# ─────────────────────────────────────────────────────────────

In [ ]:
import gzip
import json
from pathlib import Path

import duckdb
import pandas as pd
import plotly.graph_objects as go

ROOT       = Path("./..")
PARQUET    = ROOT / "data" / "parquet"
DUCKDB     = ROOT / "data" / "pestiexpo.duckdb"
GEOJSON_GZ = PARQUET / "communes_geo.geojson.gz"

STATES = {
    "no_calendar": ("#78909C", "Hors calendrier"),
    "no_data":     ("#CFD8DC", "Données manquantes"),
    0:             ("#A5D6A7", "Aucun traitement"),
    1:             ("#FFF176", "Risque faible"),
    2:             ("#FFB300", "Risque modéré"),
    3:             ("#F57C00", "Risque élevé"),
    4:             ("#B71C1C", "Risque très élevé"),
}
STATE_ORDER = ["no_calendar", "no_data", 0, 1, 2, 3, 4]

In [ ]:
# ── 1. Communes (lat/lon + has_calendar_data) ─────────────────
communes = pd.read_parquet(PARQUET / "communes_admin.parquet")
communes = communes.rename(columns={
    "code_insee":     "code_insee",
    "nom_commune":    "nom_commune",
    "code_insee_dep": "code_insee_dep",
})
print(f"{len(communes)} communes chargées")

In [ ]:
# ── 2. Risque pour la date choisie (depuis DuckDB) ────────────
con = duckdb.connect(str(DUCKDB), read_only=True)

tables = [t[0] for t in con.execute("SHOW TABLES").fetchall()]

# Essai historique d'abord, puis prévisions
df_risque = pd.DataFrame()
if "risque_journalier" in tables:
    df_risque = con.execute("""
        SELECT insee_com, risque_0_4, ift_journalier_total,
               interdiction_pulv, pluie_limitante, risque_dispersion
        FROM risque_journalier
        WHERE date = ?
    """, [DATE]).df()

if df_risque.empty and "risque_previsions" in tables:
    df_risque = con.execute("""
        SELECT insee_com, risque_0_4, ift_journalier_total,
               interdiction_pulv, pluie_limitante, risque_dispersion
        FROM risque_previsions
        WHERE date = ?
    """, [DATE]).df()
    print("⚡ Données prévisionnelles")

con.close()
print(f"{len(df_risque)} communes avec données de risque pour le {DATE}")

In [ ]:
# ── 3. IFT communes (has_calendar_data) ───────────────────────
# On récupère les colonnes culture pour déterminer has_calendar_data
import sys
sys.path.insert(0, str(ROOT / "etl"))
from etl_config import CULTURE_MAPPING
import polars as pl

ift = pl.read_parquet(PARQUET / "ift_communes_enrichi.parquet")
cal = pl.read_parquet(PARQUET / "calendrier_epandage.parquet")
old, new = list(CULTURE_MAPPING.keys()), list(CULTURE_MAPPING.values())

ift = ift.with_columns([
    pl.col("c_maj").replace_strict(old=old, new=new, default=None).alias("c_maj_cal"),
    pl.col("c_ift_hbc").replace_strict(old=old, new=new, default=None).alias("c_ift_hbc_cal"),
    pl.col("c_ift_h").replace_strict(old=old, new=new, default=None).alias("c_ift_h_cal"),
])

cal_pairs = set(zip(
    cal["departement_code"].cast(pl.Utf8).to_list(),
    cal["culture"].to_list(),
))

def _has_cal(row):
    dep = str(row.get("code_insee_dep") or "")
    for col in ("c_maj_cal", "c_ift_hbc_cal", "c_ift_h_cal"):
        val = row.get(col)
        if pd.notna(val) and (dep, val) in cal_pairs:
            return True
    return False

ift_pd = ift.select(["insee_com", "code_insee_dep", "c_maj_cal", "c_ift_hbc_cal", "c_ift_h_cal"]).to_pandas()
ift_pd["has_calendar_data"] = ift_pd.apply(_has_cal, axis=1)
print(f"{ift_pd['has_calendar_data'].sum()} communes dans le calendrier")

In [ ]:
# ── 4. Assemblage ─────────────────────────────────────────────
map_data = (
    communes
    .merge(ift_pd[["insee_com", "has_calendar_data"]], left_on="code_insee", right_on="insee_com", how="left")
    .merge(df_risque, left_on="code_insee", right_on="insee_com", how="left")
)
map_data["has_calendar_data"] = map_data["has_calendar_data"].fillna(False)

def get_state(row):
    if not row["has_calendar_data"]:
        return "no_calendar"
    if pd.isna(row.get("risque_0_4")):
        return "no_data"
    return int(row["risque_0_4"])

map_data["state"] = map_data.apply(get_state, axis=1)
map_data["color"] = map_data["state"].map(lambda s: STATES.get(s, STATES["no_data"])[0])
map_data["label"] = map_data["state"].map(lambda s: STATES.get(s, STATES["no_data"])[1])

# Résumé
counts = map_data["state"].value_counts()
print(f"Date : {DATE}")
for s in STATE_ORDER:
    clr, lbl = STATES[s]
    print(f"  {lbl:35s}: {counts.get(s, 0):6d}")

In [ ]:
# ── 5. Chargement GeoJSON ─────────────────────────────────────
with gzip.open(GEOJSON_GZ, "rt", encoding="utf-8") as f:
    geojson = json.load(f)
print(f"{len(geojson['features'])} géométries chargées")

In [ ]:
# ── 6. Carte Plotly ───────────────────────────────────────────
state_to_int = {s: i for i, s in enumerate(STATE_ORDER)}
n = len(STATE_ORDER)

colorscale = []
for i, s in enumerate(STATE_ORDER):
    c = STATES[s][0]
    colorscale.append([i / n, c])
    colorscale.append([(i + 1) / n, c])

map_data["state_int"] = map_data["state"].map(state_to_int).fillna(1)
map_data["hover"] = map_data.apply(
    lambda r: (
        f"<b>{r['nom_commune']}</b> ({r['code_insee']})<br>"
        f"Dép. {r['code_insee_dep']}<br>"
        + (f"Indicateur : {int(r['risque_0_4'])}<br>" if pd.notna(r.get('risque_0_4')) else "")
        + r["label"]
    ),
    axis=1,
)

insee_to_state = dict(zip(map_data["code_insee"], map_data["state_int"]))
insee_to_hover = dict(zip(map_data["code_insee"], map_data["hover"]))

locations  = [f["properties"]["code_insee"] for f in geojson["features"]]
z          = [insee_to_state.get(loc, 1) for loc in locations]
hovertext  = [insee_to_hover.get(loc, loc) for loc in locations]

fig = go.Figure()

fig.add_trace(go.Choroplethmap(
    geojson=geojson,
    locations=locations,
    z=z,
    colorscale=colorscale,
    zmin=0, zmax=n - 1,
    marker_opacity=0.75,
    marker_line_width=0,
    hovertext=hovertext,
    hovertemplate="%{hovertext}<extra></extra>",
    featureidkey="properties.code_insee",
    showscale=False,
))

# Légende
for s in STATE_ORDER:
    clr, lbl = STATES[s]
    fig.add_trace(go.Scattermap(
        lat=[None], lon=[None], mode="markers",
        marker=dict(size=10, color=clr),
        name=lbl, showlegend=True,
    ))

fig.update_layout(
    title=f"Carte de risque — {DATE}",
    map=dict(
        style="carto-positron",
        center=dict(lat=46.5, lon=2.3),
        zoom=5,
    ),
    height=750,
    margin=dict(l=0, r=0, t=40, b=0),
    legend=dict(
        orientation="v", x=0.01, y=0.99,
        bgcolor="rgba(255,255,255,0.9)",
        bordercolor="#ccc", borderwidth=1,
    ),
)

fig.show()